In [1]:
import pandas as pd
import numpy as np

df_orders = pd.read_csv("../dados/olist_orders_dataset.csv")
df_items = pd.read_csv("../dados/olist_order_items_dataset.csv")
df_products = pd.read_csv("../dados/olist_products_dataset.csv")
df_customers = pd.read_csv("../dados/olist_customers_dataset.csv")
df_payments = pd.read_csv("../dados/olist_order_payments_dataset.csv")
df_reviews = pd.read_csv("../dados/olist_order_reviews_dataset.csv")

In [2]:


date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_orders:df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")
df_orders.info()


df_items["shipping_limit_date"] = pd.to_datetime(df_items["shipping_limit_date"],errors="coerce")

df_items.info()


date_cols_reviews = ["review_creation_date","review_answer_timestamp"]

for col in date_cols_reviews:df_reviews[col] = pd.to_datetime(df_reviews[col], errors="coerce")

df_reviews.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype     

In [3]:
df_fato = df_items.merge(
    df_orders,
    on="order_id",
    how="left")

df_fato.head()

df_fato["faturamento"] = df_fato["price"] + df_fato["freight_value"]

df_fato[["order_id", "price", "freight_value", "faturamento"]].head()



df_fato = df_fato.merge(
    df_products,
    on="product_id",
    how="left"
)


df_fato.info()
df_fato.isnull().sum().sort_values(ascending=False).head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  object        
 3   seller_id                      112650 non-null  object        
 4   shipping_limit_date            112650 non-null  datetime64[ns]
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   customer_id                    112650 non-null  object        
 8   order_status                   112650 non-null  object        
 9   order_purchase_timestamp       112650 non-null  datetime64[ns]
 10  order_approved_at              112635 non-null  datetime64[ns]
 11  

order_delivered_customer_date    2454
product_description_lenght       1603
product_name_lenght              1603
product_category_name            1603
product_photos_qty               1603
order_delivered_carrier_date     1194
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_weight_g                   18
dtype: int64

In [4]:
dim_cliente = df_customers[["customer_id","customer_city","customer_state"]].drop_duplicates()

#dim_cliente.head()

dim_produto = df_products[["product_id","product_category_name"]].drop_duplicates()

#dim_produto.head()

dim_pedido = df_orders[[ "order_id","order_status", "order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date"]].drop_duplicates()

dim_pagamento = df_payments[["order_id","payment_type","payment_installments","payment_value"]].drop_duplicates()

#dim_pagamento.head()


dim_avaliacao = df_reviews[["order_id","review_score"]].drop_duplicates()

dim_cliente.shape
dim_produto.shape
dim_pedido.shape
dim_pagamento.shape
dim_avaliacao.shape




(98875, 2)

In [5]:
faturamento_total = df_items["price"].sum()
faturamento_total

qtd_pedidos = df_orders["order_id"].nunique()
qtd_pedidos

ticket_medio = faturamento_total / qtd_pedidos
ticket_medio

frete_medio = df_items["freight_value"].mean()
frete_medio

avaliacao_media = df_reviews["review_score"].mean()
avaliacao_media


df_atrasos = df_orders.copy()

df_atrasos["order_delivered_customer_date"] = pd.to_datetime(
    df_atrasos["order_delivered_customer_date"], errors="coerce"
)
df_atrasos["order_estimated_delivery_date"] = pd.to_datetime(
    df_atrasos["order_estimated_delivery_date"], errors="coerce"
)

pedidos_entregues = df_atrasos.dropna(
    subset=["order_delivered_customer_date"]
)

pedidos_atrasados = pedidos_entregues[
    pedidos_entregues["order_delivered_customer_date"]
    > pedidos_entregues["order_estimated_delivery_date"]
]

percentual_atraso = (
    len(pedidos_atrasados) / len(pedidos_entregues)
) * 100

percentual_atraso



kpis = {
    "Faturamento Total": faturamento_total,
    "Total de Pedidos": qtd_pedidos,
    "Ticket Médio": ticket_medio,
    "Frete Médio": frete_medio,
    "Avaliação Média": avaliacao_media,
    "% Pedidos com Atraso": percentual_atraso
}

kpis





{'Faturamento Total': np.float64(13591643.7),
 'Total de Pedidos': 99441,
 'Ticket Médio': np.float64(136.68048088816482),
 'Frete Médio': np.float64(19.990319928983578),
 'Avaliação Média': np.float64(4.08642062404257),
 '% Pedidos com Atraso': 8.112898544715783}

In [6]:
df_fat_estado = (df_items.merge(df_orders[["order_id", "customer_id"]], on="order_id").merge(df_customers[["customer_id", "customer_state"]], on="customer_id").groupby("customer_state", as_index=False)["price"].sum().sort_values("price", ascending=False))

df_fat_estado.head(10)


df_fat_categoria = (df_items.merge(df_products[["product_id", "product_category_name"]], on="product_id").groupby("product_category_name", as_index=False)["price"].sum().sort_values("price", ascending=False))

df_fat_categoria.head(10)

df_pagamentos = (df_payments.groupby("payment_type", as_index=False)["order_id"].count().sort_values("order_id", ascending=False))

df_pagamentos


df_reviews_orders = (df_reviews.merge(df_orders, on="order_id"))

df_reviews_orders["atrasado"] = (pd.to_datetime(df_reviews_orders["order_delivered_customer_date"], errors="coerce")> pd.to_datetime(df_reviews_orders["order_estimated_delivery_date"], errors="coerce"))

df_avaliacao_atraso = (df_reviews_orders.groupby("atrasado", as_index=False)["review_score"].mean())

df_avaliacao_atraso



,atrasado,review_score
0,False,4.214307
1,True,2.566550


In [9]:
fato_items = (df_items.groupby("order_id", as_index=False).agg({"price": "sum","freight_value": "sum"}))


fato_payments = (df_payments.groupby("order_id", as_index=False).agg({"payment_value": "sum"}))

fato_reviews = (df_reviews.groupby("order_id", as_index=False).agg({"review_score": "mean"}))

fato_pedidos = (df_orders.merge( df_items[["order_id", "product_id", "price", "freight_value"]],on="order_id",how="left").merge(fato_payments, on="order_id", how="left").merge(fato_reviews, on="order_id", how="left"))


fato_pedidos["atraso"] = (pd.to_datetime(fato_pedidos["order_delivered_customer_date"], errors="coerce")> pd.to_datetime(fato_pedidos["order_estimated_delivery_date"], errors="coerce"))

fato_pedidos.info()
fato_pedidos.head()



fato_pedidos.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  object        
 1   customer_id                    113425 non-null  object        
 2   order_status                   113425 non-null  object        
 3   order_purchase_timestamp       113425 non-null  datetime64[ns]
 4   order_approved_at              113264 non-null  datetime64[ns]
 5   order_delivered_carrier_date   111457 non-null  datetime64[ns]
 6   order_delivered_customer_date  110196 non-null  datetime64[ns]
 7   order_estimated_delivery_date  113425 non-null  datetime64[ns]
 8   product_id                     112650 non-null  object        
 9   price                          112650 non-null  float64       
 10  freight_value                  112650 non-null  float64       
 11  

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'product_id', 'price', 'freight_value', 'payment_value', 'review_score',
       'atraso'],
      dtype='object')

In [47]:
dim_cliente = df_customers[["customer_id","customer_city","customer_state"]].drop_duplicates()

dim_produto = df_products[["product_id","product_category_name","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]].drop_duplicates()

dim_pagamento = df_payments[["order_id","payment_type","payment_installments"]].drop_duplicates()

dim_tempo = fato_pedidos[["order_purchase_timestamp"]].drop_duplicates()
dim_tempo["date"] = pd.to_datetime(dim_tempo["order_purchase_timestamp"])
dim_tempo["ano"] = dim_tempo["date"].dt.year
dim_tempo["mes"] = dim_tempo["date"].dt.month
dim_tempo["trimestre"] = dim_tempo["date"].dt.quarter

fato_pedidos.to_csv(
    "../output/fato_pedidos.csv",
    index=False
)

dim_cliente.to_csv("../output/dim_cliente.csv", index=False)
dim_produto.to_csv("../output/dim_produto.csv", index=False)
dim_pagamento.to_csv("../output/dim_pagamento.csv", index=False)
dim_tempo.to_csv("../output/dim_tempo.csv", index=False)


